# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

In [ ]:
# List all available record sets with their @id and field @ids
record_sets_info = []
for record_set in metadata.record_sets:
    print(f"RecordSet: {record_set['@id']} - {record_set.get('name', '')}")
    field_ids = [f["@id"] for f in record_set.get("fields", [])]
    print(f"  Fields (@id): {field_ids}")
    record_sets_info.append({'id': record_set['@id'], 'field_ids': field_ids})

## 3. Data Extraction
Load data from available record sets into pandas DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Gather all record set @ids
record_sets = [rs['id'] for rs in record_sets_info]
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns in {record_set_id}: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records loaded for {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, or grouping data. This helps prepare the data for further analysis.

In [ ]:
# For demonstration, select the first non-empty record set with numeric fields
selected_record_set_id = None
numeric_field_id = None
group_field_id = None
for rs in record_sets_info:
    rset_id = rs['id']
    df = dataframes.get(rset_id)
    if df is not None and not df.empty:
        # Try to find a likely numeric field (float or int columns)
        numeric_candidates = df.select_dtypes(include=['float', 'int']).columns
        if len(numeric_candidates) > 0:
            selected_record_set_id = rset_id
            numeric_field_id = numeric_candidates[0]
            # Find first non-numeric column for grouping
            group_candidates = df.select_dtypes(exclude=['float', 'int']).columns
            if len(group_candidates) > 0:
                group_field_id = group_candidates[0]
            break

if selected_record_set_id and numeric_field_id:
    print(f"Using RecordSet: {selected_record_set_id}, Numeric field: {numeric_field_id}, Group field: {group_field_id}")
    df = dataframes[selected_record_set_id].copy()

    # Drop NA from numeric field
    filtered_df = df.dropna(subset=[numeric_field_id])
    # For demo, filter values greater than threshold
    threshold = filtered_df[numeric_field_id].mean() if not filtered_df.empty else 0
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the selected numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std())
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the group_field_id and compute means
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        display(grouped_df.head())
else:
    print('No suitable record set with numeric field was found. Please inspect dataframes variable for possible options.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and numeric_field_id:
    # Histogram of the normalized field
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), kde=True)
    plt.title(f'Normalized distribution of {numeric_field_id}')
    plt.xlabel(f'{numeric_field_id} (normalized)')
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in filtered_df.columns:
        # Boxplot grouped by the group field
        plt.figure(figsize=(12,6))
        # Show top 10 most frequent categories only for clarity
        top_values = filtered_df[group_field_id].value_counts().nlargest(10).index
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df[filtered_df[group_field_id].isin(top_values)])
        plt.title(f'{numeric_field_id} by {group_field_id} (Top 10)')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field data available for visualization.')

## 6. Conclusion
In this notebook, we explored the Mass Spectrometry Analysis dataset using the `mlcroissant` library.

**Key Observations:**
- We loaded dataset metadata and reviewed the available record sets and their field `@id`s.
- Data from the main record sets was extracted and analyzed in DataFrames.
- We demonstrated filtering and normalization on numeric fields, and grouping (where possible) for summary insights.
- Simple visualizations were created to understand the numeric field's distribution and its variation across a grouping field.

You can use the provided code pattern to perform deeper analysis or integrate the dataset into your own ML or data science pipeline. Refer to the data overview for available fields and consult the Croissant schema for precise metadata semantics.